John S. Anvik, 2026  
University of Lethbridge Student ID: 001224816

In [29]:
library(tidyverse)
library(tidymodels)
library(rpart.plot) # for plotting decision tree
library(vip) # for ranking feature importance
library(ranger) # engine for random forest

# set random seed
set.seed(42)

In [30]:

data_list <- list()

# TO RUN MODEL ON ALL 11 FOCAL MALES:
folder_path <- "C:\\Users\\johna\\OneDrive\\Documents\\Birdsong\\data\\by_id"
files = list.files(folder_path, full.names=TRUE)

for (file in files) {
    data_list <- append(data_list, list(read_csv(file)))
}

Rows: 2169 Columns: 363
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr    (2): ID, attention_target_id
dbl  (360): UST, time_rel_sunrise, D_M1_M2, D_M1_M3, D_M1_M4, D_M1_M5, D_M1_...
dttm   (1): DateTime

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.
Rows: 556 Columns: 365
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr    (2): ID, attention_target_id
dbl  (362): UST, time_rel_sunrise, D_M1_M2, D_M1_M3, D_M1_M4, D_M1_M5, D_M1_...
dttm   (1): DateTime

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.
Rows: 1033 Columns: 365
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr    (2): ID, attention_target_id
dbl  (362): UST, time

In [31]:
focal_ids <- list()
for (i in seq_along(data_list)) {
    id <- data_list[[i]] %>% pull(ID) %>% unique()
    # print(id)
    focal_ids[[id]] <- id
}
# focal_ids <- list("YKY")
names(data_list) <- focal_ids
# names(data_list) <- "YKY"

In [32]:
# In full dataset, how many of each song type for the focal male?

MODEL_TYPE <- "Baseline Classifier"

accuracy <- tibble()

for (id in focal_ids) {    n_ust <- data_list[[id]] %>%
        group_by(UST) %>%
        summarize(n = n()) %>%
        mutate( ID=id, estimated_accuracy = n/nrow(data_list[[id]]),model=MODEL_TYPE) %>%
        arrange(desc(estimated_accuracy)) %>%
        rename(Top_UST = UST) %>%
        slice(1)
    accuracy <- accuracy %>% bind_rows(n_ust)
    # print(n_ust)
}
accuracy <- accuracy %>% select(ID, estimated_accuracy, model)
print(accuracy)

filename <- paste0("data/results/", MODEL_TYPE, "_test_accuracies.csv")
print(filename)
write_csv(accuracy, filename)

# A tibble: 11 × 3
   ID     estimated_accuracy model              
   <chr>               <dbl> <chr>              
 1 DgTbO              0.184  Baseline Classifier
 2 KLgK               0.138  Baseline Classifier
 3 KRDg               0.131  Baseline Classifier
 4 LbOW               0.161  Baseline Classifier
 5 OPR                0.208  Baseline Classifier
 6 RbDgY              0.141  Baseline Classifier
 7 RRO                0.255  Baseline Classifier
 8 TbTbLg             0.0992 Baseline Classifier
 9 TbWY               0.172  Baseline Classifier
10 WLpTb              0.114  Baseline Classifier
11 YKY                0.0933 Baseline Classifier
[1] "data/results/Baseline Classifier_test_accuracies.csv"


In [33]:
MODEL_TYPE <- "N greater than 30 Baseline Classifier"
min_n <- 30

accuracy <- tibble()

for (id in focal_ids) {
    ust_counts <- data_list[[id]] %>%
        group_by(UST) %>%
        summarize(n = n())
    top_n_ust <- ust_counts %>% filter(n > min_n) %>% pull(UST)
    # print(top_n_ust)
    n_ust <- data_list[[id]] %>% filter(UST %in% top_n_ust) %>%
        group_by(UST) %>%
        summarize(n = n()) %>%
        mutate(ID = id, estimated_accuracy = n/nrow(data_list[[id]] %>% filter(UST %in% top_n_ust)),model=MODEL_TYPE) %>%
        arrange(desc(estimated_accuracy)) %>%
        rename(Top_UST = UST) %>%
        slice(1)
    accuracy <- accuracy %>% bind_rows(n_ust)
    # print(n_ust)
}
accuracy <- accuracy %>% select(ID, estimated_accuracy, model)
print(accuracy)

filename <- paste0("data/results/", MODEL_TYPE, "_test_accuracies.csv")
print(filename)
write_csv(accuracy, filename)

# A tibble: 11 × 3
   ID     estimated_accuracy model                                
   <chr>               <dbl> <chr>                                
 1 DgTbO               0.19  N greater than 30 Baseline Classifier
 2 KLgK                0.226 N greater than 30 Baseline Classifier
 3 KRDg                0.159 N greater than 30 Baseline Classifier
 4 LbOW                0.170 N greater than 30 Baseline Classifier
 5 OPR                 0.237 N greater than 30 Baseline Classifier
 6 RbDgY               0.182 N greater than 30 Baseline Classifier
 7 RRO                 0.294 N greater than 30 Baseline Classifier
 8 TbTbLg              0.118 N greater than 30 Baseline Classifier
 9 TbWY                0.194 N greater than 30 Baseline Classifier
10 WLpTb               0.176 N greater than 30 Baseline Classifier
11 YKY                 0.108 N greater than 30 Baseline Classifier
[1] "data/results/N greater than 30 Baseline Classifier_test_accuracies.csv"


In [34]:
MODEL_TYPE <- "Top 5 Classes Baseline Classifier"
N <- 5

accuracy <- tibble()

for (id in focal_ids) {
    ust_counts <- data_list[[id]] %>%
        group_by(UST) %>%
        summarize(n = n()) %>%
        arrange(desc(n))
    top_n_ust <- ust_counts %>% slice(0:5) %>% pull(UST)
    # print(top_n_ust)
    n_ust <- data_list[[id]] %>% filter(UST %in% top_n_ust) %>%
        group_by(UST) %>%
        summarize(n = n()) %>%
        mutate(ID = id, estimated_accuracy = n/nrow(data_list[[id]] %>% filter(UST %in% top_n_ust)),model=MODEL_TYPE) %>%
        arrange(desc(estimated_accuracy)) %>%
        rename(Top_UST = UST) %>%
        slice(1)
    accuracy <- accuracy %>% bind_rows(n_ust)
    # print(n_ust)
}
accuracy <- accuracy %>% select(ID, estimated_accuracy, model)
print(accuracy)

filename <- paste0("data/results/", MODEL_TYPE, "_test_accuracies.csv")
print(filename)
write_csv(accuracy, filename)


# A tibble: 11 × 3
   ID     estimated_accuracy model                            
   <chr>               <dbl> <chr>                            
 1 DgTbO               0.291 Top 5 Classes Baseline Classifier
 2 KLgK                0.288 Top 5 Classes Baseline Classifier
 3 KRDg                0.288 Top 5 Classes Baseline Classifier
 4 LbOW                0.237 Top 5 Classes Baseline Classifier
 5 OPR                 0.312 Top 5 Classes Baseline Classifier
 6 RbDgY               0.276 Top 5 Classes Baseline Classifier
 7 RRO                 0.381 Top 5 Classes Baseline Classifier
 8 TbTbLg              0.237 Top 5 Classes Baseline Classifier
 9 TbWY                0.254 Top 5 Classes Baseline Classifier
10 WLpTb               0.257 Top 5 Classes Baseline Classifier
11 YKY                 0.236 Top 5 Classes Baseline Classifier
[1] "data/results/Top 5 Classes Baseline Classifier_test_accuracies.csv"


In [35]:
top_n_ust

[1]  21   1   9 166  27